# 📊 EduPro Online Platform — Predictive Modeling Analysis
### Phase 1: Exploratory Data Analysis (EDA)
**Objective:** Understand course demand patterns, revenue drivers, and instructor impact to build predictive models.

#### 🔷 Imports & Settings

In [ ]:
# ── Imports & Settings ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style="whitegrid", palette="Set2")

print("✅ All libraries loaded successfully!")

#### 🔷 Load Datasets

In [ ]:
# ── Load Datasets ───────────────────────────────────────────────────
BASE_PATH = r"C:\Users\Rohit\Downloads\Unified Mentor\project\data"

courses      = pd.read_csv(f"{BASE_PATH}\\Courses.csv")
teachers     = pd.read_csv(f"{BASE_PATH}\\Teachers.csv")
transactions = pd.read_csv(f"{BASE_PATH}\\Transactions.csv")
users        = pd.read_csv(f"{BASE_PATH}\\Users.csv")

# Parse date column
transactions['TransactionDate'] = pd.to_datetime(
    transactions['TransactionDate'], dayfirst=True
)

print(f"✅ Courses      : {courses.shape}")
print(f"✅ Teachers     : {teachers.shape}")
print(f"✅ Transactions : {transactions.shape}")
print(f"✅ Users        : {users.shape}")

#### 🔷 Dataset Overview

In [ ]:
# ── Dataset Overview ────────────────────────────────────────────────
for name, df in [("COURSES", courses), ("TEACHERS", teachers),
                 ("TRANSACTIONS", transactions), ("USERS", users)]:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  Shape   : {df.shape[0]} rows × {df.shape[1]} cols")
    print(f"  Columns : {list(df.columns)}")
    print(f"  Nulls   : {df.isnull().sum().sum()} total missing values")
    print(f"  Duplicates: {df.duplicated().sum()}")

#### 🔷 Descriptive Statistics

In [ ]:
# ── Descriptive Statistics ──────────────────────────────────────────
print("📌 COURSES — Numerical Summary")
display(courses.describe().round(2))

print("\n📌 TEACHERS — Numerical Summary")
display(teachers.describe().round(2))

print("\n📌 TRANSACTIONS — Numerical Summary")
display(transactions.describe().round(2))

#### 🔷 Build Master DataFrame

In [ ]:
# ── Build Master DataFrame ──────────────────────────────────────────
# Enrollment count per course
enrollment = (transactions
              .groupby('CourseID')
              .size()
              .reset_index(name='EnrollmentCount'))

# Revenue per course
revenue = (transactions
           .groupby('CourseID')['Amount']
           .sum()
           .reset_index(name='CourseRevenue'))

# Merge everything
master = (courses
          .merge(enrollment, on='CourseID', how='left')
          .merge(revenue,    on='CourseID', how='left'))

master['EnrollmentCount'] = master['EnrollmentCount'].fillna(0).astype(int)
master['CourseRevenue']   = master['CourseRevenue'].fillna(0).round(2)

# Category-level revenue
cat_revenue = (master
               .groupby('CourseCategory')['CourseRevenue']
               .sum()
               .reset_index(name='CategoryRevenue'))

master = master.merge(cat_revenue, on='CourseCategory', how='left')

print("✅ Master DataFrame created!")
print(f"   Shape: {master.shape}")
display(master.head(5))

#### 🔷 EDA

##### EDA 1: Enrollment Distribution

In [ ]:
# ── EDA 1: Enrollment Distribution ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(master['EnrollmentCount'], bins=15,
             color='steelblue', edgecolor='white', linewidth=0.8)
axes[0].set_title('Enrollment Count Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Enrollments per Course')
axes[0].set_ylabel('Number of Courses')
axes[0].axvline(master['EnrollmentCount'].mean(), color='red',
                linestyle='--', label=f"Mean: {master['EnrollmentCount'].mean():.0f}")
axes[0].legend()

# Top 10 courses by enrollment
top10_enroll = master.nlargest(10, 'EnrollmentCount')
axes[1].barh(top10_enroll['CourseName'], top10_enroll['EnrollmentCount'],
             color='steelblue', edgecolor='white')
axes[1].set_title('Top 10 Courses by Enrollment', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Enrollment Count')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(f"{BASE_PATH}\\..\\notebooks\\plot_enrollment_distribution.png", dpi=150)
plt.show()
print(f"\n📌 Avg Enrollments : {master['EnrollmentCount'].mean():.1f}")
print(f"📌 Max Enrollments : {master['EnrollmentCount'].max()} — {master.loc[master['EnrollmentCount'].idxmax(), 'CourseName']}")
print(f"📌 Min Enrollments : {master['EnrollmentCount'].min()} — {master.loc[master['EnrollmentCount'].idxmin(), 'CourseName']}")

##### EDA 2: Enrollment by Category & Level

In [ ]:
# ── EDA 2: Enrollment by Category & Level ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# By Category
cat_enroll = (master.groupby('CourseCategory')['EnrollmentCount']
              .sum().sort_values(ascending=False))
axes[0].bar(cat_enroll.index, cat_enroll.values,
            color=sns.color_palette("Set2", len(cat_enroll)))
axes[0].set_title('Total Enrollments by Category', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Total Enrollments')
axes[0].tick_params(axis='x', rotation=45)

# By Level
level_enroll = master.groupby('CourseLevel')['EnrollmentCount'].sum()
colors = ['#4CAF50', '#2196F3', '#FF5722']
axes[1].pie(level_enroll.values, labels=level_enroll.index,
            autopct='%1.1f%%', colors=colors,
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Enrollment Share by Course Level', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{BASE_PATH}\\..\\notebooks\\plot_enrollment_category_level.png", dpi=150)
plt.show()

##### EDA 3: Revenue Analysis

In [ ]:
# ── EDA 3: Revenue Analysis ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Revenue by category
cat_rev_sorted = cat_revenue.sort_values('CategoryRevenue', ascending=True)
bars = axes[0].barh(cat_rev_sorted['CourseCategory'],
                    cat_rev_sorted['CategoryRevenue'],
                    color=sns.color_palette("Set2", len(cat_rev_sorted)))
axes[0].set_title('Total Revenue by Category', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Revenue (₹)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'₹{x/1000:.0f}K'))

# Top 10 courses by revenue
top10_rev = master[master['CourseRevenue'] > 0].nlargest(10, 'CourseRevenue')
axes[1].barh(top10_rev['CourseName'], top10_rev['CourseRevenue'],
             color='coral', edgecolor='white')
axes[1].set_title('Top 10 Courses by Revenue', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Revenue (₹)')
axes[1].invert_yaxis()
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'₹{x/1000:.0f}K'))

plt.tight_layout()
plt.savefig(f"{BASE_PATH}\\..\\notebooks\\plot_revenue_analysis.png", dpi=150)
plt.show()

print(f"\n📌 Total Platform Revenue : ₹{master['CourseRevenue'].sum():,.2f}")
print(f"📌 Paid Courses Revenue   : ₹{master[master['CourseType']=='Paid']['CourseRevenue'].sum():,.2f}")
print(f"📌 Free Courses Revenue   : ₹{master[master['CourseType']=='Free']['CourseRevenue'].sum():,.2f}")

##### EDA 4: Free vs Paid

In [ ]:
# ── EDA 4: Free vs Paid ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Count
type_count = master['CourseType'].value_counts()
axes[0].pie(type_count.values, labels=type_count.index,
            autopct='%1.1f%%', colors=['#2196F3', '#FF9800'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Free vs Paid — Course Count', fontsize=13, fontweight='bold')

# Avg Enrollment
type_enroll = master.groupby('CourseType')['EnrollmentCount'].mean()
axes[1].bar(type_enroll.index, type_enroll.values,
            color=['#2196F3', '#FF9800'], edgecolor='white', width=0.5)
axes[1].set_title('Avg Enrollment: Free vs Paid', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Average Enrollments')
for i, v in enumerate(type_enroll.values):
    axes[1].text(i, v + 1, f'{v:.1f}', ha='center', fontweight='bold')

# Avg Revenue (paid only makes sense)
type_rev = master.groupby('CourseType')['CourseRevenue'].mean()
axes[2].bar(type_rev.index, type_rev.values,
            color=['#2196F3', '#FF9800'], edgecolor='white', width=0.5)
axes[2].set_title('Avg Revenue: Free vs Paid', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Average Revenue (₹)')
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'₹{x/1000:.0f}K'))
for i, v in enumerate(type_rev.values):
    axes[2].text(i, v + 100, f'₹{v:,.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(f"{BASE_PATH}\\..\\notebooks\\plot_free_vs_paid.png", dpi=150)
plt.show()

##### EDA 5: Correlation Analysis

In [ ]:
# ── EDA 5: Correlation Analysis ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Price vs Enrollment
axes[0].scatter(master['CoursePrice'], master['EnrollmentCount'],
                alpha=0.7, color='steelblue', s=80, edgecolors='white')
z = np.polyfit(master['CoursePrice'], master['EnrollmentCount'], 1)
p = np.poly1d(z)
x_line = np.linspace(master['CoursePrice'].min(), master['CoursePrice'].max(), 100)
axes[0].plot(x_line, p(x_line), "r--", linewidth=2, label='Trend')
axes[0].set_title('Price vs Enrollment', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Course Price (₹)')
axes[0].set_ylabel('Enrollment Count')
axes[0].legend()

# Rating vs Enrollment
axes[1].scatter(master['CourseRating'], master['EnrollmentCount'],
                alpha=0.7, color='coral', s=80, edgecolors='white')
z2 = np.polyfit(master['CourseRating'], master['EnrollmentCount'], 1)
p2 = np.poly1d(z2)
x_line2 = np.linspace(master['CourseRating'].min(), master['CourseRating'].max(), 100)
axes[1].plot(x_line2, p2(x_line2), "r--", linewidth=2, label='Trend')
axes[1].set_title('Rating vs Enrollment', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Course Rating')
axes[1].set_ylabel('Enrollment Count')
axes[1].legend()

# Duration vs Enrollment
axes[2].scatter(master['CourseDuration'], master['EnrollmentCount'],
                alpha=0.7, color='mediumseagreen', s=80, edgecolors='white')
z3 = np.polyfit(master['CourseDuration'], master['EnrollmentCount'], 1)
p3 = np.poly1d(z3)
x_line3 = np.linspace(master['CourseDuration'].min(), master['CourseDuration'].max(), 100)
axes[2].plot(x_line3, p3(x_line3), "r--", linewidth=2, label='Trend')
axes[2].set_title('Duration vs Enrollment', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Course Duration (hrs)')
axes[2].set_ylabel('Enrollment Count')
axes[2].legend()

plt.tight_layout()
plt.savefig(f"{BASE_PATH}\\..\\notebooks\\plot_correlations.png", dpi=150)
plt.show()

##### EDA 6: Instructor Impact

In [ ]:
# ── EDA 6: Instructor Impact ─────────────────────────────────────────
# Join transactions → courses → teachers
trans_detail = (transactions
                .merge(courses[['CourseID','CourseName','CourseCategory']], on='CourseID')
                .merge(teachers[['TeacherID','TeacherName','YearsOfExperience',
                                  'TeacherRating','Expertise']], on='TeacherID'))

teacher_perf = (trans_detail
                .groupby('TeacherID')
                .agg(
                    TotalEnrollments = ('CourseID','count'),
                    TotalRevenue     = ('Amount','sum'),
                    TeacherRating    = ('TeacherRating','first'),
                    YearsOfExperience= ('YearsOfExperience','first')
                ).reset_index())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Experience vs Enrollments
axes[0].scatter(teacher_perf['YearsOfExperience'],
                teacher_perf['TotalEnrollments'],
                alpha=0.7, color='mediumpurple', s=80, edgecolors='white')
axes[0].set_title("Instructor Experience vs Enrollments", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Years of Experience")
axes[0].set_ylabel("Total Enrollments")

# Teacher Rating vs Revenue
axes[1].scatter(teacher_perf['TeacherRating'],
                teacher_perf['TotalRevenue'],
                alpha=0.7, color='tomato', s=80, edgecolors='white')
axes[1].set_title("Instructor Rating vs Revenue Generated", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Teacher Rating")
axes[1].set_ylabel("Total Revenue (₹)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'₹{x/1000:.0f}K'))

plt.tight_layout()
plt.savefig(f"{BASE_PATH}\\..\\notebooks\\plot_instructor_impact.png", dpi=150)
plt.show()

##### EDA 7: Transaction Trends Over Time

In [ ]:
# ── EDA 7: Transaction Trends Over Time ─────────────────────────────
transactions['Month'] = transactions['TransactionDate'].dt.to_period('M')

monthly = (transactions
           .groupby('Month')
           .agg(Enrollments=('TransactionID','count'),
                Revenue=('Amount','sum'))
           .reset_index())
monthly['Month'] = monthly['Month'].astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(monthly['Month'], monthly['Enrollments'],
             marker='o', color='steelblue', linewidth=2, markersize=5)
axes[0].set_title('Monthly Enrollment Trend', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Enrollments')
axes[0].tick_params(axis='x', rotation=45)
axes[0].fill_between(range(len(monthly)), monthly['Enrollments'],
                     alpha=0.15, color='steelblue')
axes[0].set_xticks(range(len(monthly)))
axes[0].set_xticklabels(monthly['Month'], rotation=45, ha='right')

axes[1].plot(monthly['Month'], monthly['Revenue'],
             marker='s', color='coral', linewidth=2, markersize=5)
axes[1].set_title('Monthly Revenue Trend', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Revenue (₹)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].fill_between(range(len(monthly)), monthly['Revenue'],
                     alpha=0.15, color='coral')
axes[1].set_xticks(range(len(monthly)))
axes[1].set_xticklabels(monthly['Month'], rotation=45, ha='right')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'₹{x/1000:.0f}K'))

plt.tight_layout()
plt.savefig(f"{BASE_PATH}\\..\\notebooks\\plot_time_trends.png", dpi=150)
plt.show()

##### EDA 8: Correlation Heatmap

In [ ]:
# ── EDA 8: Correlation Heatmap ───────────────────────────────────────
numeric_cols = ['CoursePrice', 'CourseDuration', 'CourseRating',
                'EnrollmentCount', 'CourseRevenue']

corr_matrix = master[numeric_cols].corr()

plt.figure(figsize=(9, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, center=0, linewidths=0.8,
            annot_kws={'size': 12, 'weight': 'bold'})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(f"{BASE_PATH}\\..\\notebooks\\plot_heatmap.png", dpi=150)
plt.show()

##### EDA 9: User Demographics

In [ ]:
# ── EDA 9: User Demographics ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Age distribution
axes[0].hist(users['Age'], bins=15, color='steelblue',
             edgecolor='white', linewidth=0.8)
axes[0].set_title('User Age Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Number of Users')
axes[0].axvline(users['Age'].mean(), color='red', linestyle='--',
                label=f"Mean Age: {users['Age'].mean():.1f}")
axes[0].legend()

# Gender split
gender_count = users['Gender'].value_counts()
axes[1].pie(gender_count.values, labels=gender_count.index,
            autopct='%1.1f%%', colors=['#2196F3', '#E91E63'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('User Gender Distribution', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{BASE_PATH}\\..\\notebooks\\plot_user_demographics.png", dpi=150)
plt.show()

####  📝 Phase 1 — Key EDA Insights

In [ ]:
## 📝 Phase 1 — Key EDA Insights

### Enrollment
- Enrollment is fairly uniform across courses (range: 140–196), suggesting equal platform promotion
- **[Fill after running]** Top category by enrollment: ___
- Course Level split is roughly balanced across Beginner / Intermediate / Advanced

### Revenue
- Free courses account for significant enrollments but ₹0 revenue — `CourseType` is a critical feature
- **AI, Business, and Project Management** are top revenue-generating categories
- Marketing category generates ₹0 — all courses may be free

### Instructor Impact
- **[Fill after running]** Instructor experience correlation with enrollment: ___
- **[Fill after running]** Higher-rated teachers tend to generate ___ revenue

### Correlations
- **[Fill after running]** Strongest predictor of EnrollmentCount: ___
- **[Fill after running]** Strongest predictor of CourseRevenue: ___

### Time Trends
- **[Fill after running]** Peak enrollment month: ___
- **[Fill after running]** Revenue trend: growing / stable / declining

In [ ]:
# ── Save Master DF for Phase 2 (Feature Engineering) ────────────────
SAVE_PATH = r"C:\Users\Rohit\Downloads\Unified Mentor\project\data\master_eda.csv"
master.to_csv(SAVE_PATH, index=False)
print(f"✅ Master DataFrame saved → {SAVE_PATH}")
print(f"   Shape: {master.shape}")
print(f"\n📌 Columns available for Phase 2:")
for col in master.columns:
    print(f"   → {col}")